## Libraries

In [ ]:
import os 
import json
import unicodedata
from collections import defaultdict
from docx import Document

## Normalizar

In [6]:
def normalizar_text(texto):
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ASCII', 'ignore').decode('utf-8')
    texto = texto.upper().strip()
    return " ".join(texto.split())

## Extraer Proyectos

In [ ]:
def extract_partido(path_docx, carpeta_salida):
    os.makedirs(carpeta_salida, exist_ok=True)

    doc = Document(path_docx)
    partido_actual = None
    resultados = []

    for table in doc.tables:
        for row in table.rows:
            celdas = [cell.text.strip() for cell in row.cells]

            # Detectar nombre del partido (después de la palabra "PARTIDO")
            if any("PARTIDO" in cel.upper() for cel in celdas):
                partido_actual = " ".join(
                    cel.replace("PARTIDO", "").strip() for cel in celdas if "PARTIDO" in cel.upper()
                )
                continue

            # 🚫 Ignorar encabezados como "Nº" o "PROYECTO DE LEY"
            if not celdas or len(celdas) < 4:
                continue
            if "PROYECTO DE LEY" in " ".join(celdas).upper() or celdas[0].upper() == "Nº":
                continue

            # Extraer datos si hay un partido definido
            if partido_actual:
                resultado = {
                    "partido": partido_actual,
                    "proyectoLeyNumero": celdas[1],   # celdas[0] es "Nº"
                    "tema": celdas[2],
                    "autores": celdas[3],
                    "observaciones": celdas[4] if len(celdas) > 4 else ""
                }
                resultados.append(resultado)

    # 🗂 Agrupar por partido normalizado
    agrupado_por_partido = defaultdict(list)
    for item in resultados:
        partido = normalizar_text(item["partido"])
        proyecto = {
            "proyectoLeyNumero": item["proyectoLeyNumero"],
            "tema": item["tema"],
            "autores": item["autores"],
            "observaciones": item["observaciones"]
        }
        agrupado_por_partido[partido].append(proyecto)

    # 💾 Guardar JSON agrupado
    nombre_base = os.path.splitext(os.path.basename(path_docx))[0]
    ruta_json = os.path.join(carpeta_salida, f"{nombre_base}_proyectos_agrupado.json")

    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(agrupado_por_partido, f, indent=4, ensure_ascii=False)

    print(f"✅ Proyectos agrupados guardados en: {ruta_json}")

## Uses

In [ ]:
carpeta_docx = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\Proyectos_ley.docx"
carpeta_salida = r"C:\Users\juans\Documents\pro\Model-Extract-information\document\salida_partidos_2021"

extract_partido(carpeta_docx, carpeta_salida)  

✅ Proyectos agrupados guardados en: C:\Users\juans\Documents\pro\Model-Extract-information\document\salida_partidos_2021\Proyectos_ley_proyectos_agrupado.json
